<a href="https://colab.research.google.com/github/RajManish8340/gpt-shakespeare/blob/main/gpt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [29]:
# import libraries
import torch
import torch.nn as nn
from torch.nn import functional as F

In [30]:
# dataset
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

with open('input.txt' ,'r' , encoding='utf-8') as f:
  text = f.read()

print("number of characters",len(text))
print(text[:200])


--2026-08-04 19:14:24--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt.14’

input.txt.14        100%[===================>]   1.06M  --.-KB/s    in 0.06s   

2026-08-04 19:14:24 (17.1 MB/s) - ‘input.txt.14’ saved [1115394/1115394]

number of characters 1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you


In [31]:
#hyper params
batch_size = 16
block_size = 32
max_iters = 5000
eval_interval = 100
learning_rate = 1e-3
eval_iters = 200
n_embed = 64
n_head = 4
n_layer = 4
dropout = 0.0
device = "cuda" if torch.cuda.is_available() else "cpu"
#------------------

torch.manual_seed(1337) # for matching output form the video

In [32]:
#unique chars
chars = sorted(list(set(text)))
vocab = len(chars)
print(''.join((chars)))
print(vocab)

# string to integer and integer to string
stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join(itos[i] for i in l)

print(encode("hello"))
print(decode(encode("hello")))



 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65
[46, 43, 50, 50, 53]
hello


In [33]:
# train and test splits
data = torch.tensor(encode(text) , dtype=torch.long)
print(data.shape , data.dtype)
print(data[:100])
n = int((0.9*len(data)))
train_data = data[:n]
val_data = data[n:]

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


In [34]:
# data loading
def get_batch(split):
  data = train_data if split == "train" else val_data
  ix = torch.randint(len(data) - block_size , (batch_size,))
  x = torch.stack([data[i:i+block_size] for i in ix])
  y = torch.stack([data[i+1:i+block_size+1] for i in ix])
  x, y = x.to(device) , y.to(device)
  return x, y
